# Laguna de Bay sensor placement — reproducible N=20 reference

**Status:** Proposed **working computational reference** for review; the historical 25-cell notebook remains archived and immutable. This notebook is designed to run top-to-bottom with no cached research artifacts or external account.

**Question:** Under the archived lake dataset, which **5 of 20 free candidate stations** minimize the original notebook-derived quadratic objective, given 15 explicitly marked existing stations and **local one-hop** adjacency?

This notebook deliberately does **not** reproduce the manuscript's Table V inputs exactly; see [paper-to-code discrepancies](../../../docs/WATER_NOTEBOOK_SOURCE_AUDIT.md) and issue #16. It uses the frozen **notebook weights**, not the published Table IV weights. There is no QPU, QAOA or quantum-advantage claim. All coordinates are in **EPSG:32651 metres**.

**Run:** From the repository root (Python 3.12): `python -m pip install -r problems/water_quality/notebooks/requirements-n20.txt` then open this notebook in Jupyter with the repository root as the working directory. CI uses a fresh kernel and recomputes everything.

**Research boundaries:** N=20 / K=5 is a **small validated baseline**, not a replacement for historical N=50–2000 benchmark results or SA/grid-search protocols. Those larger experiments must remain separately preregistered and run with proper computational budgets.

## 1 · Provenance gate

Fail immediately if the archived notebook or GeoJSON source bytes differ. The original notebook may contain stale/mismatched caches, so we do **not** execute or load its pickle artifacts here.

This notebook is a *new cold implementation* grounded in documented notebook Cells 2–5 and the independent reference tests, not a textual reformatting of the legacy 25 cells.

In [ ]:
from pathlib import Path
import hashlib
import json
import platform
from itertools import combinations

import numpy as np
from scipy.spatial import cKDTree
from shapely import concave_hull
from shapely.geometry import MultiPoint, Point, LineString

ROOT = Path.cwd().resolve()
DATA_PATH = ROOT / "problems/water_quality/data/LDB_centroids_clean.geojson"
ORIGINAL_NOTEBOOK_PATH = ROOT / "problems/water_quality/notebooks/experiment9 grid search copy.ipynb"
EXPECTED_DATA_SHA = "e303cce3d6af6ff4162bb31e09ca27215bc4e29b164fa0181e3f6931de3ae3a7"
EXPECTED_NOTEBOOK_SHA = "08cbc1572048f4c34aab508ae195c82575acd685c7d29695d0eccecdf67cc048"

def verified_sha(path: Path, expected: str) -> str:
    if not path.is_file():
        raise FileNotFoundError(f"Run the notebook from the repository root; missing: {path}")
    actual = hashlib.sha256(path.read_bytes()).hexdigest()
    if actual != expected:
        raise ValueError(f"Archived source changed: {path.name} has SHA-256 {actual}")
    return actual

data_sha = verified_sha(DATA_PATH, EXPECTED_DATA_SHA)
notebook_sha = verified_sha(ORIGINAL_NOTEBOOK_PATH, EXPECTED_NOTEBOOK_SHA)
print("Archived lake data verified:", data_sha)
print("Archived source notebook verified:", notebook_sha)

## 2 · Explicit scientific settings

These eight names map directly to the committed GeoJSON columns. The **D2 field is road proximity**, not the manuscript's claimed stakeholder/population-exposure criterion. The weight vector is the one hardcoded in the archived notebook's Cell 2.

The objective uses a redundancy length of 7.5 km, a wake decay length of 1 km, eastward current, and a one-hop adjacency radius derived from fixed-station spacing. These are **not** interchangeable with long-range connected-network guarantees.

In [ ]:
FACTOR_NAMES = (
    "a1_river_proximity", "a2_runoff_proximity", "b1_fishpen_density",
    "b2_hypoxic_proximity", "c1_bathymetric_depressions",
    "c2_outlet_proximity", "d1_boatramp_proximity", "d2_road_proximity",
)
WEIGHTS = np.array([0.3671, 0.1224, 0.2508, 0.0502, 0.1278, 0.0183, 0.0476, 0.0159])
N_FREE, K_NEW = 20, 5
L_C_M, L_W_M = 7500.0, 1000.0
D_MAX_BUFFER = 1.15
HULL_RATIO, HEX_POOL_SIZE = 0.05, 10000
WAKE_DIRECTION_EAST = (1.0, 0.0)
BETA, DELTA = 1.0, 1.0
assert len(FACTOR_NAMES) == len(WEIGHTS) == 8
assert np.isfinite(WEIGHTS).all() and (WEIGHTS >= 0).all()
print("N =", N_FREE, "| K =", K_NEW, "| 8 notebook factor weights =", WEIGHTS.tolist())

## 3 · Read the actual source data

The source GeoJSON has 5,417 features; station identifiers in this notebook are **zero-based feature positions**, not the `id` property. The data CRS is projected metres (EPSG:32651). Do not infer lake stations from plot coordinates without authoritative geospatial validation.

In [ ]:
raw = json.loads(DATA_PATH.read_text(encoding="utf-8"))
features = raw["features"]
coords = np.asarray([f["geometry"]["coordinates"][:2] for f in features], dtype=float)
factors = np.asarray([
    [np.nan if f["properties"].get(name) is None else float(f["properties"][name])
     for name in FACTOR_NAMES]
    for f in features
], dtype=float)
existing_indices = np.flatnonzero([
    bool(f["properties"].get("has_existing_station", False))
    for f in features
])
assert len(features) == 5417 and coords.shape == (5417, 2)
assert np.isfinite(coords).all() and len(existing_indices) == 15
print("Feature count:", len(features), "| Marked existing stations:", len(existing_indices))

## 4 · Site utility: missing-value-aware AHP weighting

For site `i`, the normalized utility is the weighted sum over *available* criteria divided by the sum of their corresponding weights; missing criteria do not become zero-risk observations.

```text
u_i = clip( sum_{k observed} w_k * x_ik / sum_{k observed} w_k , 0, 1 )
```

The printed manuscript's main-criteria AHP matrix and reported weights disagree (issue #16); do **not** replace this weight vector without explicitly defining a separate sensitivity experiment.

In [ ]:
valid = ~np.isnan(factors)
weighted_sum = np.nansum(factors * WEIGHTS, axis=1)
available_weight = np.nansum(valid * WEIGHTS, axis=1)
utility = np.clip(
    weighted_sum / np.where(available_weight == 0.0, 1.0, available_weight),
    0.0, 1.0,
)
assert np.isfinite(utility).all() and len(utility) == len(features)
print("Utility range:", float(utility.min()), float(utility.max()))

## 5 · Construct 20 spatially distributed free candidates

The archived notebook's CELL 3 uses a **10,000-site approximately hexagonal pool** across the source bounding box, clips it to a concave lake hull, snaps to master-site positions, removes existing stations, and selects candidate sites by farthest-point sampling (FPS) seeded from those fixed stations.

This is spatial *candidate selection*, not the optimization outcome. Do not replace it with an arbitrary top-20 utility ranking: that would change the original instance.

In [ ]:
hull = concave_hull(MultiPoint(coords), ratio=HULL_RATIO, allow_holes=True)
xmin, ymin = np.min(coords, axis=0)
xmax, ymax = np.max(coords, axis=0)
dx = float(np.sqrt((xmax - xmin) * (ymax - ymin) / HEX_POOL_SIZE / (np.sqrt(3) / 2)))
dy = dx * np.sqrt(3) / 2
hex_pool = [
    (xmin + col * dx + (dx / 2 if row % 2 else 0), ymin + row * dy)
    for row in range(int(np.ceil((ymax - ymin) / dy)) + 1)
    for col in range(int(np.ceil((xmax - xmin) / dx)) + 1)
    if ymin + row * dy <= ymax
    and xmin + col * dx + (dx / 2 if row % 2 else 0) <= xmax
]
water_hex = np.asarray([p for p in hex_pool if hull.contains(Point(p))], dtype=float)
assert len(water_hex) > 0, "The spatial analysis failed to produce candidate water sites"
snapped_master = cKDTree(coords).query(water_hex, k=1)[1]
remaining = np.unique(snapped_master[~np.isin(snapped_master, existing_indices)])
assert len(remaining) >= N_FREE
min_distance = cKDTree(coords[existing_indices]).query(coords[remaining], k=1)[0]
free_indices = []
for _ in range(N_FREE):
    best_position = int(np.argmax(min_distance))
    chosen_master = int(remaining[best_position])
    free_indices.append(chosen_master)
    min_distance = np.minimum(
        min_distance, np.linalg.norm(coords[remaining] - coords[chosen_master], axis=1)
    )
    min_distance[best_position] = -1.0
assert len(set(free_indices)) == N_FREE
print("N=20 candidate master-feature positions:", free_indices)

## 6 · Original quadratic objective and one-hop constraints

Let `x_i∈{0,1}` select the `i`th new candidate. We **minimize**
`Σ_i a_i x_i + Σ_(i<j) q_ij x_i x_j`,
where `a_i` incorporates negative utility and pair interactions with already-fixed stations, and `q_ij` penalizes free–free redundancy and directional wake overlap.

**Budget:** `Σ_i x_i = 5` (exact, not at most).

**Local adjacency:** `x_i ≤ f_i + Σ_(j∈N(i)) x_j` for every free candidate, where `f_i` indicates a nearby fixed station. This local rule is *not* global network path connectivity.

Edges crossing outside the derived lake hull are excluded. Here `β=δ=1`; altering them creates a new experiment.

In [ ]:
positions = np.vstack([coords[free_indices], coords[existing_indices]])
fixed_local = set(range(N_FREE, len(positions)))
nearest_fixed = cKDTree(coords[existing_indices]).query(
    coords[existing_indices], k=2
)[0][:, 1]
D_MAX_M = float(np.max(nearest_fixed) * D_MAX_BUFFER)
linear = -utility[free_indices].copy()
quadratic = []
free_neighbors = [set() for _ in range(N_FREE)]
fixed_neighbor = np.zeros(N_FREE, dtype=int)
for i, j in combinations(range(len(positions)), 2):
    p, q = positions[i], positions[j]
    if not (hull.contains(Point(p)) and hull.contains(Point(q))
            and hull.contains(LineString([p, q]))):
        continue
    displacement = q - p
    distance = float(np.linalg.norm(displacement))
    if distance == 0:
        continue
    if distance <= D_MAX_M:
        if i < N_FREE and j < N_FREE:
            free_neighbors[i].add(j)
            free_neighbors[j].add(i)
        elif i < N_FREE and j in fixed_local:
            fixed_neighbor[i] = 1
        elif j < N_FREE and i in fixed_local:
            fixed_neighbor[j] = 1
    if distance > 2 * L_C_M:
        continue
    redundancy = max(0.0, 1.0 - distance / L_C_M)
    cosine = float(np.clip(displacement[0] / distance, -1.0, 1.0))
    wake_forward = np.exp(-distance / L_W_M) * cosine if cosine > 0.7071 else 0.0
    wake_backward = -np.exp(-distance / L_W_M) * cosine if -cosine > 0.7071 else 0.0
    coefficient = float(BETA * redundancy + DELTA * (wake_forward + wake_backward))
    if abs(coefficient) <= 1e-12:
        continue
    if i < N_FREE and j < N_FREE:
        quadratic.append((i, j, coefficient))
    elif i < N_FREE:
        linear[i] += coefficient
    elif j < N_FREE:
        linear[j] += coefficient
neighbors = tuple(tuple(sorted(group)) for group in free_neighbors)
assert len(quadratic) == 12
print("One-hop maximum distance (m):", D_MAX_M)
print("Nonzero free–free interactions:", len(quadratic))

## 7 · Independent exact baseline (15,504 combinations)

`C(20,5) = 15,504` possible five-site selections fit within a bounded exhaustive check. This lets us validate the small instance *independently of SCIP and simulated annealing*.

An enumeration is feasible for **N=20 only**. It is not a viable algorithm for the N=2000 historical tier.

In [ ]:
def original_evaluation(selected):
    chosen = set(int(i) for i in selected)
    one_hop_violations = [
        int(i) for i in chosen
        if not fixed_neighbor[i] and not chosen.intersection(neighbors[i])
    ]
    energy = float(sum(float(linear[i]) for i in chosen) + sum(
        float(weight) for i, j, weight in quadratic if i in chosen and j in chosen
    ))
    return {
        "energy": energy,
        "budget_ok": len(chosen) == K_NEW,
        "one_hop_ok": not one_hop_violations,
        "violated_local_indices": sorted(one_hop_violations),
    }

best_energy = float("inf")
best_sets = []
tested = 0
for candidate in combinations(range(N_FREE), K_NEW):
    tested += 1
    assessment = original_evaluation(candidate)
    if not assessment["one_hop_ok"]:
        continue
    energy = assessment["energy"]
    if energy < best_energy - 1e-9:
        best_energy, best_sets = energy, [candidate]
    elif abs(energy - best_energy) <= 1e-9:
        best_sets.append(candidate)

assert tested == 15504 and best_sets
print("Evaluated candidates:", tested)
print("Certified enumeration best original objective:", best_energy)
print("Best feasible local selection(s):", best_sets)

## 8 · SCIP MILP implementation and independent feasibility check

SCIP solves an **equivalent mixed-integer linearization** of this quadratic binary model. Auxiliary variables `p_ij = x_i x_j` satisfy `p_ij ≥ x_i+x_j−1`, `p_ij ≤ x_i`, `p_ij ≤ x_j`. SCIP uses the *same* budget and local one-hop rules as the enumeration.

**This is not the original notebook's exact JijModeling/OMMX compiler** and no equivalence is asserted for a different historical input. We demand SCIP's explicit `optimal` status; a time-limited feasible incumbent is not an optimum.

In [ ]:
from pyscipopt import Model, quicksum

scip = Model("notebook_derived_laguna_n20")
scip.hideOutput()
x = [scip.addVar(name=f"x{i}", vtype="B") for i in range(N_FREE)]
pair_variables = []
for i, j, _ in quadratic:
    z = scip.addVar(name=f"p{i}_{j}", vtype="B")
    scip.addCons(z >= x[i] + x[j] - 1)
    scip.addCons(z <= x[i])
    scip.addCons(z <= x[j])
    pair_variables.append(z)
scip.addCons(quicksum(x) == K_NEW, name="budget")
for i in range(N_FREE):
    scip.addCons(
        x[i] <= int(fixed_neighbor[i]) + quicksum(x[j] for j in neighbors[i]),
        name=f"local_one_hop_{i}",
    )
objective = quicksum(float(linear[i]) * x[i] for i in range(N_FREE))
objective += quicksum(
    float(q) * pair_variables[k] for k, (_, _, q) in enumerate(quadratic)
)
scip.setObjective(objective, sense="minimize")
scip.setRealParam("limits/time", 30.0)
scip.optimize()
scip_status = str(scip.getStatus())
if scip_status != "optimal":
    raise RuntimeError(f"SCIP did not certify an optimal solution: {scip_status}")
sol = scip.getBestSol()
scip_selection = [
    i for i in range(N_FREE) if scip.getSolVal(sol, x[i]) > 0.5
]
scip_original = original_evaluation(scip_selection)
assert scip_original["budget_ok"] and scip_original["one_hop_ok"]
assert abs(scip_original["energy"] - best_energy) < 1e-7
print("SCIP certified status:", scip_status)
print("Selected free-site local positions:", scip_selection)
print("Independently evaluated original objective:", scip_original["energy"])

## 9 · Frozen source-derived N=20 canaries

These checks freeze a **notebook-derived** benchmark that was independently reconstructed in draft scientific tests. They are **not** a correction to manuscript Table V.

The published SCIP N=20 energy `−3.301319` differs from this source-cold result `−3.3012934918564643`; issue #16 retains the original discrepancy. Only the notebook-derived model can become our *working* N20 reference after the CI evidence is reviewed.

In [ ]:
EXPECTED_FREE = [
    5253, 2394, 1894, 1557, 3225, 4623, 47, 5409, 3922, 2408,
    2694, 4136, 2567, 1164, 4973, 4537, 2726, 968, 16, 1762,
]
EXPECTED_EXISTING = [
    417, 454, 725, 1055, 1076, 1709, 2771, 3402, 3520, 3529,
    3732, 3886, 4613, 4868, 5038,
]
EXPECTED_ENERGY = -3.3012934918564643
assert free_indices == EXPECTED_FREE
assert existing_indices.tolist() == EXPECTED_EXISTING
assert scip_selection == [0, 1, 3, 6, 7]
assert tuple(scip_selection) in best_sets
assert abs(best_energy - EXPECTED_ENERGY) < 1e-7
assert abs(scip_original["energy"] - EXPECTED_ENERGY) < 1e-7
print("N20 notebook-derived fixture matched independently verified site selection and energy.")

## 10 · Map of selected and existing stations

This chart plots **projected** positions in metres and distinguishes the computed selections from historical existing stations. It is not a bathymetric/risk heat map, a safety certification, or a global radio-connectivity visualization.

In [ ]:
import matplotlib.pyplot as plt

figure, ax = plt.subplots(figsize=(8, 6))
ax.scatter(coords[free_indices, 0], coords[free_indices, 1],
           s=25, color="#7d8590", label="20 candidates", zorder=1)
ax.scatter(coords[existing_indices, 0], coords[existing_indices, 1],
           s=35, color="#d9483b", label="15 existing stations", zorder=2)
selected_master = np.asarray(free_indices)[scip_selection]
ax.scatter(coords[selected_master, 0], coords[selected_master, 1],
           s=90, marker="*", color="#1767bb", label="5 new stations", zorder=3)
ax.set(title="Laguna de Bay: verified source-derived N=20 / K=5 placement",
       xlabel="Easting (m, EPSG:32651)", ylabel="Northing (m, EPSG:32651)")
ax.set_aspect("equal", adjustable="box")
ax.legend(loc="best")
ax.grid(alpha=0.2)
figure.tight_layout()
plt.show()
plt.close(figure)

## 11 · Machine-readable result and interpretation

The result records source identity, environment versions, **complete N20 original-model coefficients and local constraints**, selected *local* and master feature positions, and the SCIP-certified objective. Every floating-point number in the model is serialized with `float.hex()` so a cryptographic model-input digest can expose last-bit drift. A consuming test checks the digest, model invariants, and verified optimum. **An exact hash difference is not automatically a scientifically meaningful numerical difference**; float64 stability is investigated separately under issue #24.

For full provenance beyond this notebook, preserve exact compiled LP coefficients, dependency lock, complete environmental metadata and artifact checksums in a later science milestone. **Do not promote this N20 run into a generalized or publication-matching claim.**

In [ ]:
import scipy
import shapely
import pyscipopt

# Exact binary objective/constraint model (before SCIP's auxiliary MILP variables).
# Encode each float as IEEE-754 hexadecimal: rounded display numbers are not enough
# to detect one-ULP coefficient changes between runs or environments.
MODEL_INPUT = {
    "schema_version": 1,
    "model_family": "notebook_derived_laguna_N20_local_one_hop",
    "dataset_sha256": data_sha,
    "archived_notebook_sha256": notebook_sha,
    "candidate_master_feature_positions": [int(i) for i in free_indices],
    "fixed_master_feature_positions": existing_indices.astype(int).tolist(),
    "site_coords_epsg32651_float64_hex": [
        [float(v).hex() for v in row] for row in positions
    ],
    "factor_weights_float64_hex": [float(w).hex() for w in WEIGHTS],
    "budget_exact": int(K_NEW),
    "linear_float64_hex": [float(a).hex() for a in linear],
    "quadratic_ij_float64_hex": [
        [int(i), int(j), float(q).hex()] for i, j, q in quadratic
    ],
    "fixed_neighbor_flags": [int(v) for v in fixed_neighbor],
    "free_neighbor_indices": [list(members) for members in neighbors],
    "d_max_m_float64_hex": float(D_MAX_M).hex(),
    "redundancy_beta_float64_hex": float(BETA).hex(),
    "wake_delta_float64_hex": float(DELTA).hex(),
}
model_payload = (
    json.dumps(MODEL_INPUT, sort_keys=True, separators=(",", ":"),
               ensure_ascii=False, allow_nan=False) + "\n"
).encode("utf-8")
model_input_sha256 = hashlib.sha256(model_payload).hexdigest()

REFERENCE_RESULT = {
    "schema_version": 2,
    "model_input": MODEL_INPUT,
    "model_input_sha256": model_input_sha256,
    "model_input_hash_scope": "IEEE754 exact model; compare tolerance-based parity separately",
    "status": "notebook_derived_N20_replay_not_paper_parity",
    "dataset_sha256": data_sha,
    "archived_notebook_sha256": notebook_sha,
    "candidate_count": N_FREE,
    "new_station_budget": K_NEW,
    "marked_existing_stations": len(existing_indices),
    "local_one_hop_not_global_connectivity": True,
    "scip_status": scip_status,
    "selected_local_positions": scip_selection,
    "selected_master_feature_positions": selected_master.astype(int).tolist(),
    "original_energy": scip_original["energy"],
    "enumeration_energy": best_energy,
    "enumerated_combinations": tested,
    "comparison_paper_scip_energy_not_parity": -3.301319,
    "scientific_environment": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "scipy": scipy.__version__,
        "shapely": shapely.__version__,
        "pyscipopt": pyscipopt.__version__,
    },
}
print("N20_CLEAN_RESULT " + json.dumps(REFERENCE_RESULT, sort_keys=True, allow_nan=False))